# Getting Started with cande-wrapper

This notebook demonstrates how to use the `cande-wrapper` Python package to run CANDE
(Culvert ANalysis and DEsign) finite element analyses.

**Prerequisites:** The package must be installed with the compiled Fortran extension.
See `README.md` for installation instructions.

```bash
pip install -e ".[dev]"
```

## 1. Import the package

In [1]:
from cande_wrapper import CandeEngine
from pathlib import Path
import shutil
import tempfile

## 2. Set up a working directory

CANDE reads input from `.cid` files and writes output files to the same directory.
We'll copy the included example input file into a temporary working directory.

In [2]:
# Create a temporary working directory
work_dir = Path(tempfile.mkdtemp(prefix="cande_demo_"))
print(f"Working directory: {work_dir}")

# Copy the example input file
example_cid = Path("tests/example_data/MGK-IO.cid")
shutil.copy2(example_cid, work_dir / "MGK-IO.cid")

print(f"Input file: {work_dir / 'MGK-IO.cid'}")

Working directory: C:\Users\dane\AppData\Local\Temp\cande_demo_2sx6va1t
Input file: C:\Users\dane\AppData\Local\Temp\cande_demo_2sx6va1t\MGK-IO.cid


## 3. Create the engine and verify input

The `CandeEngine` takes a working directory where `.cid` files live.

In [3]:
engine = CandeEngine(work_dir=work_dir)

# Verify the input file exists
print(f"Input file exists: {engine.check_input('MGK-IO')}")

Input file exists: True


## 4. Inspect the input file

CANDE `.cid` files are fixed-format text. Let's look at the first few lines.

In [4]:
cid_text = (work_dir / "MGK-IO.cid").read_text()
for i, line in enumerate(cid_text.splitlines()[:15], 1):
    print(f"{i:3d}: {line}")

  1:                       A-1!!ANALYS   3 0  2 Bandwidth minimizer example, house on stilts                                   
  2:                    A-2.L3!!BASIC         2
  3:                 B-1.Basic!!    1    2      1000         0        10        10         0
  4:                 B-2.Basic!!    0
  5:                    A-2.L3!!BASIC         2
  6:                 B-1.Basic!!    1    2      1000         0        10        10         0
  7:                 B-2.Basic!!    0
  8:                    C-1.L3!!PREP    Non optimum node numbers                                        
  9:                    C-2.L3!!    1    4    0    3    1   11    8    5    1    0    3
 10:                    C-3.L3!!    1    0         0         0                         
 11:                    C-3.L3!!    2    0        10         0                         
 12:                    C-3.L3!!    6    0         0       2.5                         
 13:                    C-3.L3!!    7    0        10     

## 5. Run the analysis

Call `engine.run()` with the file prefix (without `.cid` extension).
This runs the CANDE Fortran engine and returns a `CandeResult` object.

> **Note:** This cell requires the compiled Fortran extension.
> If you haven't built it yet, see `README.md`.

In [5]:
result = engine.run("MGK-IO")
print(result)

CandeResult(prefix='MGK-IO', output=C:\Users\dane\AppData\Local\Temp\cande_demo_2sx6va1t\MGK-IO.out)


## 6. Inspect the results

The `CandeResult` object provides paths to all output files.

In [6]:
print(f"Output file: {result.output_file}")
print(f"Log file:    {result.log_file}")
print(f"TOC file:    {result.toc_file}")
print()

# List all files created in the working directory
print("All files in working directory:")
for f in sorted(work_dir.iterdir()):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")

Output file: C:\Users\dane\AppData\Local\Temp\cande_demo_2sx6va1t\MGK-IO.out
Log file:    C:\Users\dane\AppData\Local\Temp\cande_demo_2sx6va1t\MGK-IO.log
TOC file:    C:\Users\dane\AppData\Local\Temp\cande_demo_2sx6va1t\MGK-IO.ctc

All files in working directory:
  MGK-IO.cid (4,649 bytes)
  MGK-IO.ctc (964 bytes)
  MGK-IO.log (760 bytes)
  MGK-IO.out (24,386 bytes)
  MGK-IO_BeamResults.xml (9,471 bytes)
  MGK-IO_MeshGeom.xml (7,123 bytes)
  MGK-IO_MeshResults.xml (7,154 bytes)
  MGK-IO_PLOT1.dat (1,968 bytes)
  MGK-IO_PLOT2.dat (1,751 bytes)


## 7. Read the output report

The `.out` file contains the full CANDE analysis report.

In [7]:
output = result.output_text
lines = output.splitlines()
print(f"Total output lines: {len(lines)}")
print()
print("=== First 50 lines ===")
for line in lines[:50]:
    print(line)

Total output lines: 812

=== First 50 lines ===
>>    Input file: MGK-IO.cid
>>    Output file: MGK-IO.out
>> 
>> 
>>          *** WELCOME TO CANDE-2025 (Version April 2025) ***


           *** WELCOME TO CANDE-2025 (Version April 2025) ***



 MASTER CONTROL AND PIPE-TYPE DATA FOR PROBLEM #  1, (P#1)



          USER TITLE:  Bandwidth minimizer example, house on stilts               


          EXECUTION MODE ..................   ANALYS  

          SOLUTION LEVEL ..................  #3 USER

          METHODOLOGY (LRFD OR SERVICE) ...  SERVICE

          NUMBER OF PIPE-ELEMENT GROUPS ....       2

          MAXIMUM ITERATIONS PER STEP ......      30


>> 
>> 
>>          *** PROBLEM NUMBER  1
>> 
>> 
>>Problem title:  Bandwidth minimizer example, house on stilts
>> 
>> 
>>          EXECUTION MODE ..................   ANALYS
>> 
>>          SOLUTION LEVEL ..................  #3 USER
>> 
>>          METHODOLOGY (LRFD OR SERVICE) ...  SERVICE
>> 
>>          NUMBER OF PIPE-ELEMENT GR

In [8]:
# Check for normal completion
if "NORMAL EXIT FROM CANDE" in output:
    print("Analysis completed successfully.")
else:
    print("WARNING: Normal exit message not found in output.")

Analysis completed successfully.


## 8. Visualization with Plotly

`cande-wrapper` includes a visualization module that parses CANDE XML output files
and creates interactive Plotly figures. Install the viz extras:

```bash
pip install cande-wrapper[viz]
# or: pip install plotly
```

In [ ]:
from cande_wrapper.visualization import (
    parse_mesh_geom, parse_mesh_results, parse_beam_results,
    plot_mesh, plot_deformed_mesh, plot_beam_forces,
    plot_beam_results, plot_displacement, plot_load_history,
    plot_soil_stress,
)

# Parse the XML output files from the MGK-IO run
geom = parse_mesh_geom(result.mesh_geom_file)
mesh_results = parse_mesh_results(result.mesh_results_file)
beam = parse_beam_results(result.beam_results_file)

print(f"Mesh: {len(geom['nodes'])} nodes, {len(geom['elements'])} elements")
print(f"Element types: {set(e['type'] for e in geom['elements'])}")
print(f"Load steps: {len(mesh_results['steps'])}")
print(f"Beam groups: {len(beam['beam_groups'])}")
print(f"Beam nodes per step: {len(beam['steps'][-1]['results'])}")

### 8a. Mesh geometry with element types

The mesh plot colors elements by type: **beam** (red), **quad** (blue), **tri** (green), **interface** (gray).
Node and boundary markers are included with hover information.

In [ ]:
fig = plot_mesh(geom)
fig.show()

### 8b. Deformed mesh shape

Overlay the original (dashed) and deformed (solid) mesh with a displacement magnification factor.

In [ ]:
fig = plot_deformed_mesh(geom, mesh_results, step=-1, scale=50.0)
fig.show()

### 8c. Displacement magnitude

Color-coded displacement magnitude at each node.

In [ ]:
fig = plot_displacement(geom, mesh_results, step=-1)
fig.show()

### 8d. Beam forces (moment, thrust, shear)

Three stacked subplots showing bending moment, thrust force, and shear force
around the pipe circumference at the final load step.

In [ ]:
fig = plot_beam_forces(beam, step=-1)
fig.show()

### 8e. Individual beam quantity

Plot any single quantity around the pipe. Available: `moment`, `thrust`, `shear`,
`normal_pressure`, `tang_pressure`, `result10`-`result18`.

In [ ]:
fig = plot_beam_results(beam, step=-1, quantity="thrust")
fig.show()

### 8f. Load history

Track how forces build up through the construction/loading sequence.
Plot envelope (max/min across all nodes) or a specific node.

In [ ]:
# Thrust force envelope across all load steps
fig = plot_load_history(beam, quantity="thrust")
fig.show()

### 8g. Soil stress contours

Plot stress fields in the soil (QUAD/TRIA elements). Available components:
`vertical`, `horizontal`, `shear`, `vertical_strain`, `horizontal_strain`, `shear_strain`.

In [ ]:
fig = plot_soil_stress(geom, mesh_results, step=-1, component="vertical")
fig.show()

### 8h. Convenience methods on CandeResult

You can also call `plot_mesh()` and `plot_beam_forces()` directly on the result object:

In [ ]:
# These are shortcuts that parse + plot in one call
fig = result.plot_mesh()
fig.show()

---

# 9. Corrugated Metal Pipe Design Demo

This section demonstrates how to use `cande-wrapper` to design corrugated steel pipes
under varying fill heights — a common scenario for construction site drainage and
causeway culverts.

We'll use CANDE's **Level 1 (Elasticity) DESIGN mode with LRFD** to automatically
determine the required corrugation size and thickness for each pipe diameter and fill height.

### 9a. Define the design vehicle

CANDE uses wheel footprint geometry for live-load scaling. The `Vehicle` class provides
standard AASHTO vehicles with correct axle configurations.

In [ ]:
from cande_wrapper import Vehicle, WheelFootprint, Axle

# Standard AASHTO vehicles
hs20 = Vehicle.hs20()
hs25 = Vehicle.hs25()
tandem = Vehicle.tandem()

print(f"{'Vehicle':<15} {'Axles':<8} {'Total Weight (kip)':<20} {'Footprint (LxW)'}")
print("-" * 65)
for v in [hs20, hs25, tandem]:
    print(f"{v.name:<15} {len(v.axles):<8} {v.total_weight/1000:<20.0f} "
          f"{v.footprint.length}\" x {v.footprint.width}\"")

print(f"\nHS-20 axle layout:")
for i, axle in enumerate(hs20.axles):
    label = "Steering" if i == 0 else f"Axle {i+1}"
    print(f"  {label}: {axle.weight/1000:.0f} kip, "
          f"{'leading' if axle.spacing == 0 else f'{axle.spacing/12:.0f} ft back'}")

### 9b. Generate CID input files for multiple pipe sizes

We'll create CANDE Level 1 DESIGN problems for 24", 36", 48", and 60" diameter
corrugated steel pipes at fill heights of 2 ft, 4 ft, 8 ft, and 12 ft.

In [ ]:
def generate_cmp_design_cid(
    diameter_in: float,
    fill_height_in: float,
    title: str = "",
    n_elements: int = 61,
    lrfd: int = 1,
) -> str:
    """Generate a CANDE Level 1 DESIGN CID problem for corrugated steel pipe."""
    if not title:
        title = f"{diameter_in:.0f}in CMP, {fill_height_in/12:.0f}ft fill"

    lines = []

    # A-1: Master control - DESIGN mode, Level 1, LRFD
    a1 = f"{'A-1!!':>26}DESIGN   1 {lrfd}  {diameter_in:<5.0f} {title:<40s}          30"
    lines.append(a1)

    # A-2: Pipe type = STEEL
    lines.append(f"{'A-2.L12!!':>26}STEEL     ")

    # B-1: Steel material (PE=29e6, PNU=0.3, PYIELD=33000)
    lines.append(
        f"{'B-1.Steel!!':>26}  29000000       0.3     33000"
        f"                   0         0    0    1    0"
    )

    # B-2: LRFD design weights (all 1.0)
    lines.append(
        f"{'B-2.Steel.D.LRFD!!':>26}         1         1         1         1         1"
    )

    # B-3: LRFD resistance factors
    lines.append(
        f"{'B-3.Steel.AD.LRFD!!':>26}         1         1         1       0.9         5"
    )

    # C-1: Level 1 major input - NSTEPS(I10), HFILL(F10.0), NSOIL(I5), IPRINT(I5)
    n_layers = 10
    lines.append(
        f"{'C-1.L1!!':>26}{n_elements:>10d}{fill_height_in:>10.0f}{n_layers:>5d}    0"
    )

    # C-2: Soil zone boundaries (3, 6, 9, ..., 30)
    for i in range(1, n_layers + 1):
        sub = i * 3
        lines.append(
            f"{'C-2.L1!!':>26}{sub:>10d}{1000:>10.0f}      0.25"
        )

    # C-3: Load step factors (LRFD factor = 2.05)
    for step in range(1, n_layers + 1):
        label = f"Load step #{step}"
        lines.append(
            f"{'C-3.L1!!':>26}{step:>5d}{step:>5d}      2.05{label:<30s}"
        )

    return "\n".join(lines)


# Generate problems for multiple diameters and fill heights
diameters = [24, 36, 48, 60]  # inches
fill_heights_ft = [2, 4, 8, 12]  # feet

# Build combined CID file
cmp_dir = Path(tempfile.mkdtemp(prefix="cande_cmp_"))
cid_path = cmp_dir / "CMP-DESIGN.cid"

problems = []
for d in diameters:
    for h_ft in fill_heights_ft:
        problems.append(generate_cmp_design_cid(d, h_ft * 12))

cid_content = "\nSTOP\n".join(problems) + "\nSTOP\n"
cid_path.write_text(cid_content)

print(f"CID file: {cid_path}")
print(f"Problems: {len(problems)}")
print(f"\n--- First problem (24in CMP, 2ft fill) ---")
for i, line in enumerate(cid_content.split("STOP")[0].strip().splitlines(), 1):
    print(f"{i:3d}: {line}")

### 9c. Run all design problems

> **Note:** This cell requires the compiled Fortran extension (`pip install -e ".[dev]"`).

In [ ]:
cmp_engine = CandeEngine(work_dir=cmp_dir)
cmp_result = cmp_engine.run("CMP-DESIGN")
print(cmp_result)

if "NORMAL EXIT FROM CANDE" in cmp_result.output_text:
    print("\nAll problems completed successfully.")
else:
    print("\nWARNING: Check output for errors.")

### 9d. Parse design results

In [ ]:
import re

output_lines = cmp_result.output_text.splitlines()

print(f"{'Problem':<40s} {'Design Result'}")
print("-" * 70)

for i, line in enumerate(output_lines):
    if "USER TITLE:" in line:
        title = line.split("USER TITLE:")[-1].strip()
        design_info = ""
        for j in range(i, min(i + 200, len(output_lines))):
            if "CORRUGATION" in output_lines[j].upper() and "THICKNESS" in output_lines[j].upper():
                design_info = output_lines[j].strip()
                break
            if "SELECTED" in output_lines[j].upper():
                design_info = output_lines[j].strip()
                break
        print(f"  {title:<40s} {design_info}")

### 9e. Custom vehicle example

In [ ]:
# Define a custom heavy haul vehicle
heavy_haul = Vehicle(
    name="Heavy Haul Truck",
    footprint=WheelFootprint(length=12.0, width=24.0, spacing=84.0),
    axles=[
        Axle(weight=12000, spacing=0),      # Steering axle
        Axle(weight=40000, spacing=180),     # Drive tandem front
        Axle(weight=40000, spacing=54),      # Drive tandem rear
        Axle(weight=40000, spacing=240),     # Trailer tandem front
        Axle(weight=40000, spacing=54),      # Trailer tandem rear
    ],
)

print(f"Vehicle: {heavy_haul.name}")
print(f"Total weight: {heavy_haul.total_weight/1000:.0f} kip")
print(f"Wheel footprint: {heavy_haul.footprint.length}\" x {heavy_haul.footprint.width}\"")
print(f"\nAxle configuration:")
cumulative = 0
for i, axle in enumerate(heavy_haul.axles):
    cumulative += axle.spacing
    print(f"  Axle {i+1}: {axle.weight/1000:5.0f} kip at {cumulative/12:6.1f} ft")

---

# 10. Pipe Type Catalog

`cande-wrapper` provides Python objects for common pipe materials. Each stores the
CANDE material properties and maps to the correct CANDE pipe type
(STEEL, ALUMINUM, CONCRETE, PLASTIC, or BASIC).

In [ ]:
from cande_wrapper import (
    CorrugatedSteel, CorrugatedAluminum, CorrugatedHDPE, CorrugatedPVC,
    SmoothSteelCasing, SmoothHDPE, PrecastConcrete, DuctileIron,
)

# --- Corrugated Metal Pipes ---
cmp_steel = CorrugatedSteel()
cmp_struct_plate = CorrugatedSteel.structural_plate()
cmp_aluminum = CorrugatedAluminum()

print("=== Corrugated Metal Pipes ===")
print(f"  {cmp_steel.name:<25s} CANDE type: {cmp_steel.cande_type:<10s} E={cmp_steel.youngs_modulus/1e6:.0f} Msi, Fy={cmp_steel.yield_stress/1000:.0f} ksi")
print(f"  {cmp_struct_plate.name:<25s} CANDE type: {cmp_struct_plate.cande_type:<10s} E={cmp_struct_plate.youngs_modulus/1e6:.0f} Msi, Fy={cmp_struct_plate.yield_stress/1000:.0f} ksi")
print(f"  {cmp_aluminum.name:<25s} CANDE type: {cmp_aluminum.cande_type:<10s} E={cmp_aluminum.youngs_modulus/1e6:.0f} Msi, Fy={cmp_aluminum.yield_stress/1000:.0f} ksi")

# --- Plastic Pipes ---
hdpe_corr = CorrugatedHDPE()
pvc_corr = CorrugatedPVC()
hdpe_smooth = SmoothHDPE(diameter=24.0, wall_thickness=1.0)

print("\n=== Plastic Pipes ===")
print(f"  {hdpe_corr.name:<25s} CANDE type: {hdpe_corr.cande_type:<10s} E_short={hdpe_corr.short_term_modulus/1000:.0f} ksi, E_long={hdpe_corr.long_term_modulus/1000:.0f} ksi")
print(f"  {pvc_corr.name:<25s} CANDE type: {pvc_corr.cande_type:<10s} E_short={pvc_corr.short_term_modulus/1000:.0f} ksi, E_long={pvc_corr.long_term_modulus/1000:.0f} ksi")
print(f"  {hdpe_smooth.name:<25s} CANDE type: {hdpe_smooth.cande_type:<10s} D={hdpe_smooth.diameter}\", t={hdpe_smooth.wall_thickness}\"")

# --- Concrete Pipe ---
rcp = PrecastConcrete(diameter=48.0, wall_thickness=5.0)
rcp_c76 = PrecastConcrete.astm_c76_class_iii(diameter=48.0)

print("\n=== Concrete Pipes ===")
print(f"  {rcp.name:<25s} CANDE type: {rcp.cande_type:<10s} D={rcp.diameter}\", t={rcp.wall_thickness}\", f'c={rcp.compressive_strength} psi")
print(f"  {rcp_c76.name:<25s} CANDE type: {rcp_c76.cande_type:<10s} D={rcp_c76.diameter}\", t={rcp_c76.wall_thickness}\", Ec={rcp_c76.elastic_modulus/1e6:.2f} Msi")

# --- Smooth-Wall Steel & Ductile Iron ---
casing = SmoothSteelCasing(diameter=36.0, wall_thickness=0.375)
api_pipe = SmoothSteelCasing.api_5l_grade_b(diameter=30.0, wall_thickness=0.312)
di_pipe = DuctileIron(diameter=24.0, wall_thickness=0.33)
di_awwa = DuctileIron.awwa_c151(diameter=24.0, pressure_class=350)

print("\n=== Smooth-Wall Steel & Ductile Iron ===")
print(f"  {casing.name:<25s} CANDE type: {casing.cande_type:<10s} D={casing.diameter}\", t={casing.wall_thickness}\", Fy={casing.yield_stress/1000:.0f} ksi")
print(f"  {api_pipe.name:<25s} CANDE type: {api_pipe.cande_type:<10s} D={api_pipe.diameter}\", t={api_pipe.wall_thickness}\", Fy={api_pipe.yield_stress/1000:.0f} ksi")
print(f"  {di_pipe.name:<25s} CANDE type: {di_pipe.cande_type:<10s} D={di_pipe.diameter}\", t={di_pipe.wall_thickness}\", Fy={di_pipe.yield_stress/1000:.0f} ksi")
print(f"  {di_awwa.name:<25s} CANDE type: {di_awwa.cande_type:<10s} D={di_awwa.diameter}\", t={di_awwa.wall_thickness:.3f}\"")

# Section properties for smooth-wall pipes (BASIC type)
print("\n=== Section Properties (per unit length) ===")
for p in [casing, di_pipe, hdpe_smooth]:
    print(f"  {p.name:<25s} A={p.area_per_length:.4f} in²/in, I={p.inertia_per_length:.6f} in⁴/in")

## 11. Clean up

In [ ]:
shutil.rmtree(work_dir, ignore_errors=True)
shutil.rmtree(cmp_dir, ignore_errors=True)
print("Cleaned up temporary directories.")